In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.decomposition import DictionaryLearning
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# --- IMPORTACIÓN LIMPIA ---
try:
    from meteostat import Point, Daily
    print("Meteostat cargado correctamente.")
except ImportError as e:
    print(f"Error crítico: No se encuentra la librería. {e}")
    print("Prueba a ejecutar: pip install meteostat")
    sys.exit()

# ==========================================
# 1) EXTRACCIÓN MASIVA DE DATOS (34 AÑOS)
# ==========================================
print("Conectando a bases de datos globales (NOAA/OMM)...")

# Coordenadas de Madrid (Latitud, Longitud, Altitud)
ubicacion = Point(40.4168, -3.7038, 667)

# Definimos el periodo: 34 años (1990 - 2023)
inicio = datetime(1990, 1, 1)
fin = datetime(2023, 12, 31)

# Descargamos los datos diarios
datos_brutos = Daily(ubicacion, inicio, fin).fetch()

print(f"Datos crudos descargados: {datos_brutos.shape[0]} días.")

# ==========================================
# 2) LIMPIEZA Y PREPARACIÓN DE SERIES TEMPORALES LARGAS
# ==========================================
umbral_validos = int(len(datos_brutos) * 0.70)    # Si una columna tiene menos del 70% de datos reales se elimina
datos_limpios = datos_brutos.dropna(axis=1, thresh=umbral_validos)
datos_limpios = datos_limpios.fillna(method='ffill').fillna(method='bfill')  # Se rellenan huecos con valores posteriores y anteriores

# Extraemos la matriz puramente matemática
X_raw = datos_limpios.select_dtypes(include=[np.number])   # Nos quedamos con las columnas métricas (tª, humedad, etc, no fechas)
nombres_variables = X_raw.columns.tolist()  # Se enumeran las columnas
dias_totales = X_raw.shape[0]   # Se definen dimensiones
dimension_real = X_raw.shape[1]

print(f"Matriz consolidada: {dias_totales} días válidos con {dimension_real} dimensiones físicas.")
print("-" * 60)

# ==========================================
# 3) MÁQUINA DEL TIEMPO: ENTRENAR CON EL SIGLO XX, PREDECIR EL XXI
# ==========================================
corte = int(dias_totales * 0.70)    # Se usa el 70% de datos para predecir el último 30%
X_train_raw = X_raw.iloc[:corte]  # El antes y el después
X_test_raw = X_raw.iloc[corte:]

scaler = StandardScaler()   # Normalización de todas las variables: se invoca la función
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

print(f"Días de Entrenamiento (Pasado histórico): {X_train_scaled.shape[0]}")
print(f"Días de Prueba (La última década):        {X_test_scaled.shape[0]}")
print("-" * 60)

# ==========================================
# 4) APRENDIZAJE DEL DICCIONARIO (CON CRONÓMETRO)
# ==========================================
n_atoms = 40 
print(f"Entrenando {n_atoms} átomos... (Midiendo tiempo de ejecución)")

dict_learner = DictionaryLearning(  # Función que invoca el diccionario y forma todo el setup
    n_components=n_atoms,    
    alpha=1.0, 
    transform_algorithm='lasso_lars', 
    random_state=42,
    max_iter=1000
)

# Iniciamos el cronómetro
inicio_tiempo = time.time()

# El algoritmo aprende los patrones climáticos históricos
dict_learner.fit(X_train_scaled)   # Los datos para el train
dictionary_atoms = dict_learner.components_  # Los átomos del diccionario

# Paramos el cronómetro
fin_tiempo = time.time()
tiempo_ejecucion = fin_tiempo - inicio_tiempo

# Reconstruimos la última década entera
sparse_test = dict_learner.transform(X_test_scaled)  # Reconstrucción 
X_test_reconstructed = np.dot(sparse_test, dictionary_atoms) # Producto escalar de coeficientes por átomos aprendidos

# ==========================================
# 5) EVALUACIÓN EXACTA
# ==========================================
# MSE exacto puro, NO SPARSE CODING ERROR, respetando la regla matemática
mse_test_exacto = mean_squared_error(X_test_scaled, X_test_reconstructed)   # Se halla el MSE

ceros_exactos = np.sum(np.abs(sparse_test) < 1e-5)   
esparcidad_test = (ceros_exactos / sparse_test.size) * 100   # Se calcula el nivel de esparcidad

print("\n" + "="*60)
print(" RESULTADOS EXACTOS SOBRE 3 DÉCADAS")
print("="*60)
print(f" Tiempo de entrenamiento CPU:           {tiempo_ejecucion:.2f} segundos")
print(f" MSE Exacto en la última década (Test): {mse_test_exacto:.4f}")
print(f" Esparcidad sostenida:                  {esparcidad_test:.2f}% de ceros")
print("="*60)

# ==========================================
# 6) GRÁFICA: LA EVOLUCIÓN HISTÓRICA A LARGO PLAZO
# ==========================================
indice_temp = nombres_variables.index('tavg') if 'tavg' in nombres_variables else 0

# Suavizamos los datos (media móvil de 30 días) para que la gráfica se entienda
real_suavizado = pd.Series(X_test_scaled[:, indice_temp]).rolling(window=30).mean()
recon_suavizado = pd.Series(X_test_reconstructed[:, indice_temp]).rolling(window=30).mean()

plt.figure(figsize=(16, 6))
plt.plot(real_suavizado, color='black', label='Temp. Real (Media Móvil 30 días)', linewidth=2)
plt.plot(recon_suavizado, color='tab:red', label='Reconstrucción Matemática', linestyle='--', linewidth=2)

plt.title(f'Evaluación de Generalización a Largo Plazo (Última Década)\nExtracción Automática API | MSE Exacto: {mse_test_exacto:.4f}', fontsize=16)
plt.xlabel('Días en el set de Prueba (Aprox. 2014 - 2023)', fontsize=12)
plt.ylabel('Temperatura Estandarizada', fontsize=12)
plt.legend(loc="upper right")
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

Error crítico: No se encuentra la librería. No module named 'meteostat'
Prueba a ejecutar: pip install meteostat


SystemExit: 

In [3]:
import numpy as np
import pandas as pd
from datetime import datetime
from meteostat import Point, Daily
from sklearn.decomposition import DictionaryLearning
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1) EXTRACCIÓN MASIVA: MADRID (1990 - 2023)
# ==========================================
print("Descargando 34 años de clima de Madrid desde NOAA/OMM...")
ubicacion = Point(40.4168, -3.7038, 667)
inicio = datetime(1990, 1, 1)
fin = datetime(2023, 12, 31)

datos_brutos = Daily(ubicacion, inicio, fin).fetch()

# Limpieza estricta de sensores
umbral_validos = int(len(datos_brutos) * 0.70)
datos_limpios = datos_brutos.dropna(axis=1, thresh=umbral_validos)
datos_limpios = datos_limpios.fillna(method='ffill').fillna(method='bfill')

X_raw = datos_limpios.select_dtypes(include=[np.number])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Matriz de Madrid lista: {X_scaled.shape[0]} días válidos, {X_scaled.shape[1]} dimensiones.")
print("-" * 60)

# ==========================================
# 2) ENTRENAMIENTO DEL DICCIONARIO (EL SPARSE CODER)
# ==========================================
n_atoms = 10
lambda_param = 1.0  # Penalización L1 (alpha)

print(f"Entrenando {n_atoms} átomos históricos con Lambda = {lambda_param}...")
dict_learner = DictionaryLearning(
    n_components=n_atoms, 
    alpha=lambda_param, 
    transform_algorithm='lasso_lars', 
    random_state=42,
    max_iter=1000
)

# Aprendizaje y extracción de la matriz de coeficientes (alpha) y el diccionario (D)
sparse_representation = dict_learner.fit_transform(X_scaled)
dictionary_atoms = dict_learner.components_

# ==========================================
# 3) AUDITORÍA MATEMÁTICA: LA FUNCIÓN OBJETIVO EXACTA
# ==========================================
print("Calculando el error interno exacto de la optimización...")

# A) Término de Fidelidad (Error Residual Cuadrático Puro, sin raíces)
# Matemática: 0.5 * ||X - D * alpha||^2
X_reconstructed = np.dot(sparse_representation, dictionary_atoms)
residuales = X_scaled - X_reconstructed
error_fidelidad = 0.5 * np.sum(residuales ** 2)

# B) Término de Esparcidad (Castigo de la Norma L1)
# Matemática: Lambda * Suma absoluta de los coeficientes
penalizacion_L1 = lambda_param * np.sum(np.abs(sparse_representation))

# C) Costo Total (El mínimo global que ha encontrado tu Mac M1)
error_total_sparse = error_fidelidad + penalizacion_L1

print("\n" + "="*60)
print(" AUDITORÍA MATEMÁTICA: SPARSE CODER EN MADRID (34 AÑOS)")
print("="*60)
print(f" 1. Fidelidad (Error Cuadrático Puro): {error_fidelidad:.4f}")
print(f" 2. Castigo por átomos (Norma L1):     {penalizacion_L1:.4f}")
print("-" * 60)
print(f" -> COSTO TOTAL DEL ALGORITMO:         {error_total_sparse:.4f}")
print("="*60 + "\n")

Descargando 34 años de clima de Madrid desde NOAA/OMM...


Matriz de Madrid lista: 12418 días válidos, 5 dimensiones.
------------------------------------------------------------
Entrenando 10 átomos históricos con Lambda = 1.0...
Calculando el error interno exacto de la optimización...

 AUDITORÍA MATEMÁTICA: SPARSE CODER EN MADRID (34 AÑOS)
 1. Fidelidad (Error Cuadrático Puro): 6693.6517
 2. Castigo por átomos (Norma L1):     12447.9423
------------------------------------------------------------
 -> COSTO TOTAL DEL ALGORITMO:         19141.5940



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from meteostat import Point, Daily
from sklearn.decomposition import DictionaryLearning
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# 1. OBTENCIÓN DE DATOS (Madrid, 1990-2023)
ubicacion = Point(40.4168, -3.7038, 667)
inicio, fin = datetime(1990, 1, 1), datetime(2023, 12, 31)
data = Daily(ubicacion, inicio, fin).fetch()

# 2. LIMPIEZA RÁPIDA
data = data.dropna(axis=1, thresh=int(len(data) * 0.7))
data = data.select_dtypes(include=[np.number]).fillna(method='ffill').fillna(method='bfill')

# 3. ESCALADO
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data)

# 4. EXPERIMENTO: ÁTOMOS VS ERROR
# Probamos desde 2 hasta 100 átomos en saltos de 5 para agilizar la ejecución
rango_atomos = range(2, 101, 5)
errores = []

print("Iniciando barrido de átomos (2 a 100)...")

for n in rango_atomos:
    # Definimos el modelo
    dl = DictionaryLearning(
        n_components=n,
        transform_algorithm='lasso_lars',
        alpha=1.0,
        random_state=42,
        max_iter=500,
        n_jobs=-1 # Usamos todos los núcleos del procesador
    )
    
    # Ajustamos el diccionario
    dl.fit(X_scaled)
    
    # Representamos los datos con ese diccionario (Sparse Coding)
    X_sparse = dl.transform(X_scaled)
    
    # Reconstruimos la señal original
    X_reconstructed = np.dot(X_sparse, dl.components_)
    
    # Calculamos el error
    mse = mean_squared_error(X_scaled, X_reconstructed)
    errores.append(mse)
    print(f"Átomos: {n} | MSE: {mse:.4f}")

# 5. GRÁFICA DE CODO (ELBOW CURVE)
plt.figure(figsize=(10, 6))
plt.plot(list(rango_atomos), errores, marker='o', linestyle='-', color='#2c3e50', linewidth=2)

# Estética de la gráfica
plt.title('Optimización del Diccionario: Número de Átomos vs. Error (MSE)', fontsize=14)
plt.xlabel('Número de Átomos (Complejidad del Diccionario)', fontsize=12)
plt.ylabel('Error Cuadrático Medio (Reconstrucción)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# Resaltar el área de rendimientos decrecientes
plt.fill_between(list(rango_atomos), errores, color='skyblue', alpha=0.2)

plt.tight_layout()
plt.show()